# Notebook 04 — Model Optimization
**Thesis: Computer Vision and Deep Learning for Real-Time Quality Inspection in Manufacturing**

## Overview
Export and optimize trained models for deployment across three frameworks:
- **ONNX**: Primary CPU inference format
- **TFLite**: Edge/mobile deployment
- **Quantized** (INT8): Reduce model size + latency
- **Pruned**: Structured pruning (optional, if accuracy drop < 2%)

## Pipeline
For each trained model (YOLO + CNN × 3 datasets):
1. Load best `.pt` from Drive (`models/{yolo,cnn}/full/`)
2. Export to ONNX
3. Export to TFLite (via ONNX → TF SavedModel → TFLite)
4. Apply INT8 dynamic quantization
5. Validate exported model accuracy (≥ 95% of original)
6. Save to Drive (`models/{yolo,cnn}/{onnx,tflite,quantized}/`)

## Export Paths
```
models/yolo/onnx/{dataset}_{model}.onnx
models/yolo/tflite/{dataset}_{model}.tflite
models/yolo/quantized/{dataset}_{model}_int8.onnx
models/cnn/onnx/{dataset}_{model}.onnx
models/cnn/tflite/{dataset}_{model}.tflite
models/cnn/quantized/{dataset}_{model}_int8.onnx
```

In [ ]:
# ── Cell 1: Environment Setup ─────────────────────────────────────────────
!pip install ultralytics>=8.0.0 onnx onnxruntime onnx-tf tensorflow onnxruntime-tools -q
!pip install onnx2tf==1.17.5 -q  # for ONNX→TF→TFLite

from google.colab import drive
drive.mount('/content/drive')

import os, json, shutil, datetime
import numpy as np
DRIVE_ROOT = '/content/drive/MyDrive/thesis_project'

DATASETS = ['NEU-DET', 'DAGM', 'KSDD2']
YOLO_MODELS = ['yolov8n', 'yolov8s']
CNN_MODELS  = ['resnet50', 'efficientnet_b0', 'mobilenet_v3_large']

# Verify inputs exist
print('Checking trained model inputs...')
missing = []
for ds in DATASETS:
    for m in YOLO_MODELS:
        p = f'{DRIVE_ROOT}/models/yolo/full/{ds}_{m}_best.pt'
        if not os.path.exists(p): missing.append(p)
    for m in CNN_MODELS:
        p = f'{DRIVE_ROOT}/models/cnn/full/{ds}_{m}_best.pth'
        if not os.path.exists(p): missing.append(p)
if missing:
    print(f'WARNING: {len(missing)} model(s) not found. Run Notebooks 02 & 03 first.')
    for p in missing[:5]: print(f'  Missing: {p}')
else:
    print('All trained models found.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checking trained model inputs...
All trained models found.


In [ ]:
# ── Cell 2: YOLO Export (ONNX + TFLite + INT8) ───────────────────────────
from ultralytics import YOLO
from onnxruntime.quantization import quantize_dynamic, QuantType
import os

export_log = []

for dataset in DATASETS:
    for model_variant in YOLO_MODELS:
        run_name = f'{dataset}_{model_variant}'
        pt_path = f'{DRIVE_ROOT}/models/yolo/full/{run_name}_best.pt'

        if not os.path.exists(pt_path):
            print(f'[SKIP] Not found: {pt_path}')
            continue

        # ── ONNX FP32 ──
        onnx_out = f'{DRIVE_ROOT}/models/yolo/onnx/{run_name}.onnx'
        os.makedirs(os.path.dirname(onnx_out), exist_ok=True)
        if not os.path.exists(onnx_out):
            print(f'Exporting {run_name} → ONNX...')
            model = YOLO(pt_path)
            exported = model.export(
                format='onnx',
                imgsz=640,
                dynamic=True,
                simplify=True,
                opset=13,
            )
            # ultralytics saves next to .pt by default — copy to our dir
            default_onnx = pt_path.replace('_best.pt', '_best.onnx')
            if not os.path.exists(default_onnx):
                default_onnx = str(exported) if exported else None
            if default_onnx and os.path.exists(default_onnx):
                shutil.copy(default_onnx, onnx_out)
            elif os.path.exists(onnx_out):
                pass  # ultralytics may write directly
            size_mb = os.path.getsize(onnx_out) / 1e6 if os.path.exists(onnx_out) else 0
            print(f'  ONNX saved ({size_mb:.1f} MB): {onnx_out}')
            export_log.append({'name': run_name, 'format': 'onnx', 'size_mb': size_mb, 'path': onnx_out})
        else:
            print(f'[SKIP] ONNX already exists: {onnx_out}')

        # ── ONNX INT8 quantization ──
        quant_out = f'{DRIVE_ROOT}/models/yolo/quantized/{run_name}_int8.onnx'
        os.makedirs(os.path.dirname(quant_out), exist_ok=True)
        if os.path.exists(onnx_out) and not os.path.exists(quant_out):
            print(f'  Quantizing {run_name} → INT8...')
            quantize_dynamic(onnx_out, quant_out, weight_type=QuantType.QInt8)
            size_mb = os.path.getsize(quant_out) / 1e6
            print(f'  INT8 saved ({size_mb:.1f} MB): {quant_out}')
            export_log.append({'name': run_name, 'format': 'onnx_int8', 'size_mb': size_mb, 'path': quant_out})

        # ── TFLite FP32 ──
        tflite_out = f'{DRIVE_ROOT}/models/yolo/tflite/{run_name}.tflite'
        os.makedirs(os.path.dirname(tflite_out), exist_ok=True)
        if not os.path.exists(tflite_out):
            print(f'  Exporting {run_name} → TFLite...')
            try:
                model = YOLO(pt_path)
                model.export(format='tflite', imgsz=640, int8=False)
                # Find exported file
                import glob
                tflite_candidates = glob.glob(f'{os.path.dirname(pt_path)}/**/*.tflite', recursive=True)
                if tflite_candidates:
                    shutil.copy(tflite_candidates[0], tflite_out)
                    size_mb = os.path.getsize(tflite_out) / 1e6
                    print(f'  TFLite saved ({size_mb:.1f} MB): {tflite_out}')
                    export_log.append({'name': run_name, 'format': 'tflite', 'size_mb': size_mb, 'path': tflite_out})
            except Exception as e:
                print(f'  WARNING: TFLite export failed for {run_name}: {e}')
                export_log.append({'name': run_name, 'format': 'tflite', 'size_mb': 0, 'path': '', 'error': str(e)})

print(f'\nYOLO export complete. {len(export_log)} files processed.')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Exporting NEU-DET_yolov8n → ONNX...
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/thesis_project/models/yolo/full/NEU-DET_yolov8n_best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (6.0 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.71'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 10 packages i

  INT8 saved (3.4 MB): /content/drive/MyDrive/thesis_project/models/yolo/quantized/NEU-DET_yolov8n_int8.onnx
  Exporting NEU-DET_yolov8n → TFLite...
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/thesis_project/models/yolo/full/NEU-DET_yolov8n_best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (6.0 MB)
requirements: Ultralytics requirements ['sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0', 'onnx2tf>=1.26.3,<1.29.0'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 1.72s
Prepared 5 packages in 215ms
Uninstalled 1 package in 6ms
Installed 5 packages in 8ms
 + ai-edge-litert==2.1.2
 + backports-strenum==1.3.1
 + onnx-graphsurgeon==0.5.8
 - onnx2tf==1.17.5
 + onnx2tf==1.28.8
 + sng4onnx==2.0.0

requirements: AutoUpdate s

/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:1447: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  warnings.warn(


ONNX: slimming with onnxslim 0.1.85...
ONNX: export success ✅ 2.1s, saved as '/content/drive/MyDrive/thesis_project/models/yolo/full/NEU-DET_yolov8n_best.onnx' (11.8 MB)
Unzipping calibration_image_sample_data_20x128x128x3_float32.npy.zip to /content/calibration_image_sample_data_20x128x128x3_float32.npy...: 100% ━━━━━━━━━━━━ 1/1 36.5files/s 0.0s
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at '/content/drive/MyDrive/thesis_project/models/yolo/full/NEU-DET_yolov8n_best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 10, 8400), dtype=tf.float32, name=None)
Captures:
  138863025338704: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  138863025337168: TensorSpec(shape=(3, 3, 3, 16), dtype=tf.float32, name=None)
  138863025337936: TensorSpec(shape=(16,), dtype=tf.float32, name=Non

  INT8 saved (11.5 MB): /content/drive/MyDrive/thesis_project/models/yolo/quantized/NEU-DET_yolov8s_int8.onnx
  Exporting NEU-DET_yolov8s → TFLite...
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/thesis_project/models/yolo/full/NEU-DET_yolov8s_best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (21.5 MB)

TensorFlow SavedModel: starting export with tensorflow 2.19.0...

ONNX: starting export with onnx 1.20.1 opset 22...
ONNX: slimming with onnxslim 0.1.85...
ONNX: export success ✅ 1.8s, saved as '/content/drive/MyDrive/thesis_project/models/yolo/full/NEU-DET_yolov8s_best.onnx' (42.8 MB)
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at '/content/drive/MyDrive/thesis_project/models/yolo/full/NEU-DET_yolov8s_best_saved_model'. The following endpoints are a

  INT8 saved (3.4 MB): /content/drive/MyDrive/thesis_project/models/yolo/quantized/DAGM_yolov8n_int8.onnx
  Exporting DAGM_yolov8n → TFLite...
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/thesis_project/models/yolo/full/DAGM_yolov8n_best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (6.0 MB)

TensorFlow SavedModel: starting export with tensorflow 2.19.0...

ONNX: starting export with onnx 1.20.1 opset 22...
ONNX: slimming with onnxslim 0.1.85...
ONNX: export success ✅ 1.1s, saved as '/content/drive/MyDrive/thesis_project/models/yolo/full/DAGM_yolov8n_best.onnx' (11.8 MB)
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at '/content/drive/MyDrive/thesis_project/models/yolo/full/DAGM_yolov8n_best_saved_model'. The following endpoints are available:

* Endpoin

  INT8 saved (11.5 MB): /content/drive/MyDrive/thesis_project/models/yolo/quantized/DAGM_yolov8s_int8.onnx
  Exporting DAGM_yolov8s → TFLite...
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/thesis_project/models/yolo/full/DAGM_yolov8s_best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (21.5 MB)

TensorFlow SavedModel: starting export with tensorflow 2.19.0...

ONNX: starting export with onnx 1.20.1 opset 22...
ONNX: slimming with onnxslim 0.1.85...
ONNX: export success ✅ 1.8s, saved as '/content/drive/MyDrive/thesis_project/models/yolo/full/DAGM_yolov8s_best.onnx' (42.8 MB)
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at '/content/drive/MyDrive/thesis_project/models/yolo/full/DAGM_yolov8s_best_saved_model'. The following endpoints are available:

* End

  INT8 saved (3.4 MB): /content/drive/MyDrive/thesis_project/models/yolo/quantized/KSDD2_yolov8n_int8.onnx
  Exporting KSDD2_yolov8n → TFLite...
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/thesis_project/models/yolo/full/KSDD2_yolov8n_best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (6.0 MB)

TensorFlow SavedModel: starting export with tensorflow 2.19.0...

ONNX: starting export with onnx 1.20.1 opset 22...
ONNX: slimming with onnxslim 0.1.85...
ONNX: export success ✅ 1.2s, saved as '/content/drive/MyDrive/thesis_project/models/yolo/full/KSDD2_yolov8n_best.onnx' (11.8 MB)
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at '/content/drive/MyDrive/thesis_project/models/yolo/full/KSDD2_yolov8n_best_saved_model'. The following endpoints are available:

* En

  INT8 saved (11.5 MB): /content/drive/MyDrive/thesis_project/models/yolo/quantized/KSDD2_yolov8s_int8.onnx
  Exporting KSDD2_yolov8s → TFLite...
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/thesis_project/models/yolo/full/KSDD2_yolov8s_best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (21.5 MB)

TensorFlow SavedModel: starting export with tensorflow 2.19.0...

ONNX: starting export with onnx 1.20.1 opset 22...
ONNX: slimming with onnxslim 0.1.85...
ONNX: export success ✅ 1.8s, saved as '/content/drive/MyDrive/thesis_project/models/yolo/full/KSDD2_yolov8s_best.onnx' (42.8 MB)
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at '/content/drive/MyDrive/thesis_project/models/yolo/full/KSDD2_yolov8s_best_saved_model'. The following endpoints are available:



In [ ]:
# ── Cell 3: CNN → ONNX Export (fixed: always embeds weights, avoids tiny stub ONNX) ──
import os
import torch
import torch.onnx
import onnx
import torchvision.models as models
import torch.nn as nn

NUM_CLASSES_MAP = {'NEU-DET': 6, 'DAGM': 2, 'KSDD2': 2}

def build_cnn(model_name, num_classes):
    if model_name == 'resnet50':
        m = models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif model_name == 'efficientnet_b0':
        m = models.efficientnet_b0(weights=None)
        # torchvision efficientnet_b0: classifier = [Dropout, Linear]
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif model_name == 'mobilenet_v3_large':
        m = models.mobilenet_v3_large(weights=None)
        # torchvision mobilenet_v3_large: classifier = [Linear(960->1280), Hardswish, Dropout, Linear(1280->1000)]
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    else:
        raise ValueError(f"Unknown model_name: {model_name}")
    return m

def force_embed_onnx_weights(onnx_path: str):
    """
    If the ONNX was saved with external data or ends up as a tiny 'stub', re-save it as a single file
    with all tensors embedded.
    """
    m = onnx.load(onnx_path, load_external_data=True)
    onnx.checker.check_model(m)

    # Re-save as a single embedded ONNX file (no external tensor data)
    onnx.save_model(
        m,
        onnx_path,
        save_as_external_data=False
    )

cnn_export_log = []
dummy = torch.randn(1, 3, 224, 224, device='cpu')

for dataset in DATASETS:
    num_classes = NUM_CLASSES_MAP[dataset]
    for model_name in CNN_MODELS:
        run_name   = f'{dataset}_{model_name}'
        pth_path  = f'{DRIVE_ROOT}/models/cnn/full/{run_name}_best.pth'
        onnx_path = f'{DRIVE_ROOT}/models/cnn/onnx/{run_name}.onnx'
        os.makedirs(os.path.dirname(onnx_path), exist_ok=True)

        if os.path.exists(onnx_path):
            print(f'[SKIP] ONNX exists: {onnx_path}')
            continue
        if not os.path.exists(pth_path):
            print(f'[SKIP] Model not found: {pth_path}')
            continue

        print(f'Exporting {run_name} → ONNX...')
        model = build_cnn(model_name, num_classes).to('cpu')

        state = torch.load(pth_path, map_location='cpu')
        model.load_state_dict(state)
        model.eval()

        # Sanity check: forward output must be logits [B, num_classes]
        with torch.no_grad():
            y = model(dummy)
        if y.ndim != 2 or y.shape[1] != num_classes:
            raise RuntimeError(
                f"{run_name}: model forward shape is {tuple(y.shape)}; expected (batch, {num_classes}). "
                "Your checkpoint/model head wiring is wrong."
            )

        torch.onnx.export(
            model,
            dummy,
            onnx_path,
            opset_version=18,
            input_names=['input'],
            output_names=['output'],
            dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
            do_constant_folding=True,
            export_params=True,
            training=torch.onnx.TrainingMode.EVAL,
        )

        # Load + check + force single-file embedded weights (prevents 0.2MB stub ONNX)
        force_embed_onnx_weights(onnx_path)

        # Final check + size
        onnx_model = onnx.load(onnx_path)
        onnx.checker.check_model(onnx_model)

        size_mb = os.path.getsize(onnx_path) / 1e6
        print(f'  Saved {onnx_path} ({size_mb:.1f} MB)')
        cnn_export_log.append({'name': run_name, 'format': 'onnx', 'size_mb': size_mb})

print('CNN ONNX export complete.')


[SKIP] ONNX exists: /content/drive/MyDrive/thesis_project/models/cnn/onnx/NEU-DET_resnet50.onnx
[SKIP] ONNX exists: /content/drive/MyDrive/thesis_project/models/cnn/onnx/NEU-DET_efficientnet_b0.onnx
[SKIP] ONNX exists: /content/drive/MyDrive/thesis_project/models/cnn/onnx/NEU-DET_mobilenet_v3_large.onnx
[SKIP] ONNX exists: /content/drive/MyDrive/thesis_project/models/cnn/onnx/DAGM_resnet50.onnx
[SKIP] ONNX exists: /content/drive/MyDrive/thesis_project/models/cnn/onnx/DAGM_efficientnet_b0.onnx
[SKIP] ONNX exists: /content/drive/MyDrive/thesis_project/models/cnn/onnx/DAGM_mobilenet_v3_large.onnx
[SKIP] ONNX exists: /content/drive/MyDrive/thesis_project/models/cnn/onnx/KSDD2_resnet50.onnx
[SKIP] ONNX exists: /content/drive/MyDrive/thesis_project/models/cnn/onnx/KSDD2_efficientnet_b0.onnx
[SKIP] ONNX exists: /content/drive/MyDrive/thesis_project/models/cnn/onnx/KSDD2_mobilenet_v3_large.onnx
CNN ONNX export complete.


In [ ]:
# ── Cell: Install/repair onnx2tf dependencies ──
!pip -q uninstall -y onnx2tf
!pip -q install onnx_graphsurgeon
!pip -q install onnx2tf


In [ ]:
# ── Cell 4: CNN TFLite Conversion (ONNX → onnx2tf → TFLite) ─────────────
import os, subprocess, shutil
import tensorflow as tf

for dataset in DATASETS:
    for model_name in CNN_MODELS:
        run_name   = f'{dataset}_{model_name}'
        onnx_path  = f'{DRIVE_ROOT}/models/cnn/onnx/{run_name}.onnx'
        tflite_out = f'{DRIVE_ROOT}/models/cnn/tflite/{run_name}.tflite'
        int8_out   = f'{DRIVE_ROOT}/models/cnn/quantized/{run_name}_int8.tflite'
        saved_model_dir = f'/tmp/{run_name}_saved_model'

        os.makedirs(os.path.dirname(tflite_out), exist_ok=True)
        os.makedirs(os.path.dirname(int8_out), exist_ok=True)

        if os.path.exists(tflite_out) and os.path.exists(int8_out):
            print(f'[SKIP] TFLite exists: {tflite_out} and {int8_out}')
            continue
        if not os.path.exists(onnx_path):
            print(f'[SKIP] ONNX not found: {onnx_path}')
            continue

        print(f'Converting {run_name} → TFLite...')
        try:
            if os.path.exists(saved_model_dir):
                shutil.rmtree(saved_model_dir, ignore_errors=True)

            # Step 1: ONNX → TF SavedModel via onnx2tf
            cmd = ['onnx2tf', '-i', onnx_path, '-o', saved_model_dir, '-osd', '--non_verbose']
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=900)

            if result.returncode != 0:
                msg = (result.stderr or result.stdout or "onnx2tf failed").strip()
                raise RuntimeError(msg[-2000:])  # show last part (most useful)

            # Step 2: TF SavedModel → TFLite FP32
            converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
            tflite_model = converter.convert()
            with open(tflite_out, 'wb') as f:
                f.write(tflite_model)

            # Step 3: INT8 (weight quantization only, stable without calibration dataset)
            converter_int8 = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
            converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
            tflite_int8 = converter_int8.convert()
            with open(int8_out, 'wb') as f:
                f.write(tflite_int8)

            size_mb = os.path.getsize(tflite_out) / 1e6
            size_int8_mb = os.path.getsize(int8_out) / 1e6
            print(f'  TFLite FP32: {size_mb:.1f} MB → {tflite_out}')
            print(f'  TFLite INT8: {size_int8_mb:.1f} MB → {int8_out}')
            cnn_export_log.append({'name': run_name, 'format': 'tflite', 'size_mb': size_mb})
            cnn_export_log.append({'name': run_name, 'format': 'tflite_int8', 'size_mb': size_int8_mb})

        except Exception as e:
            print(f'  WARNING: TFLite conversion failed for {run_name}: {e}')
            cnn_export_log.append({'name': run_name, 'format': 'tflite', 'size_mb': 0, 'error': str(e)})
            cnn_export_log.append({'name': run_name, 'format': 'tflite_int8', 'size_mb': 0, 'error': str(e)})

        finally:
            if os.path.exists(saved_model_dir):
                shutil.rmtree(saved_model_dir, ignore_errors=True)

print('CNN TFLite conversion complete.')


Converting NEU-DET_resnet50 → TFLite...
  TFLite FP32: 94.0 MB → /content/drive/MyDrive/thesis_project/models/cnn/tflite/NEU-DET_resnet50.tflite
  TFLite INT8: 23.9 MB → /content/drive/MyDrive/thesis_project/models/cnn/quantized/NEU-DET_resnet50_int8.tflite
Converting NEU-DET_efficientnet_b0 → TFLite...
  TFLite FP32: 16.0 MB → /content/drive/MyDrive/thesis_project/models/cnn/tflite/NEU-DET_efficientnet_b0.tflite
  TFLite INT8: 4.5 MB → /content/drive/MyDrive/thesis_project/models/cnn/quantized/NEU-DET_efficientnet_b0_int8.tflite
Converting NEU-DET_mobilenet_v3_large → TFLite...
  TFLite FP32: 16.8 MB → /content/drive/MyDrive/thesis_project/models/cnn/tflite/NEU-DET_mobilenet_v3_large.tflite
  TFLite INT8: 4.5 MB → /content/drive/MyDrive/thesis_project/models/cnn/quantized/NEU-DET_mobilenet_v3_large_int8.tflite
Converting DAGM_resnet50 → TFLite...
  TFLite FP32: 94.0 MB → /content/drive/MyDrive/thesis_project/models/cnn/tflite/DAGM_resnet50.tflite
  TFLite INT8: 23.9 MB → /content/driv

In [ ]:
# ── Cell 5: CNN ONNX INT8 Dynamic Quantization (fixed: sanitize + correct size calc) ──
import os
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

def sanitize_onnx_remove_value_info(src_path: str, dst_path: str):
    """
    Removes internal value_info annotations that can be wrong and break ORT quantization shape inference.
    Also loads external data if present, then saves a clean single-file ONNX.
    """
    m = onnx.load(src_path, load_external_data=True)

    # Remove problematic internal annotations
    del m.graph.value_info[:]

    # Save as a single embedded ONNX file (prevents stub/external-data weirdness)
    onnx.save_model(m, dst_path, save_as_external_data=False)
    return dst_path

for dataset in DATASETS:
    for model_name in CNN_MODELS:
        run_name  = f'{dataset}_{model_name}'
        onnx_path = f'{DRIVE_ROOT}/models/cnn/onnx/{run_name}.onnx'
        quant_out = f'{DRIVE_ROOT}/models/cnn/quantized/{run_name}_int8.onnx'
        os.makedirs(os.path.dirname(quant_out), exist_ok=True)

        if os.path.exists(quant_out):
            print(f'[SKIP] INT8 exists: {quant_out}')
            continue
        if not os.path.exists(onnx_path):
            print(f'[SKIP] ONNX not found: {onnx_path}')
            continue

        print(f'Quantizing {run_name} → INT8...')
        try:
            # Sanitize to avoid ShapeInferenceError from bad internal metadata
            san_path = onnx_path.replace(".onnx", "_fp32_sanitized.onnx")
            sanitize_onnx_remove_value_info(onnx_path, san_path)

            # Quantize from the sanitized model
            quantize_dynamic(san_path, quant_out, weight_type=QuantType.QInt8)

            # Correct size comparison: use sanitized FP32 (the real full model)
            size_fp32 = os.path.getsize(san_path) / 1e6
            size_int8 = os.path.getsize(quant_out) / 1e6
            reduction = (1 - size_int8 / size_fp32) * 100

            print(f'  FP32: {size_fp32:.1f} MB → INT8: {size_int8:.1f} MB ({reduction:.0f}% smaller)')
            cnn_export_log.append({'name': run_name, 'format': 'onnx_int8', 'size_mb': size_int8})

            # Optional cleanup (uncomment if you don't want to keep sanitized copies)
            # os.remove(san_path)

        except Exception as e:
            print(f'  ERROR: {e}')

print('CNN INT8 quantization complete.')


[SKIP] INT8 exists: /content/drive/MyDrive/thesis_project/models/cnn/quantized/NEU-DET_resnet50_int8.onnx
[SKIP] INT8 exists: /content/drive/MyDrive/thesis_project/models/cnn/quantized/NEU-DET_efficientnet_b0_int8.onnx
[SKIP] INT8 exists: /content/drive/MyDrive/thesis_project/models/cnn/quantized/NEU-DET_mobilenet_v3_large_int8.onnx
[SKIP] INT8 exists: /content/drive/MyDrive/thesis_project/models/cnn/quantized/DAGM_resnet50_int8.onnx
[SKIP] INT8 exists: /content/drive/MyDrive/thesis_project/models/cnn/quantized/DAGM_efficientnet_b0_int8.onnx
[SKIP] INT8 exists: /content/drive/MyDrive/thesis_project/models/cnn/quantized/DAGM_mobilenet_v3_large_int8.onnx
[SKIP] INT8 exists: /content/drive/MyDrive/thesis_project/models/cnn/quantized/KSDD2_resnet50_int8.onnx
[SKIP] INT8 exists: /content/drive/MyDrive/thesis_project/models/cnn/quantized/KSDD2_efficientnet_b0_int8.onnx
[SKIP] INT8 exists: /content/drive/MyDrive/thesis_project/models/cnn/quantized/KSDD2_mobilenet_v3_large_int8.onnx
CNN INT8 q

In [ ]:
# ── Cell: CNN Pruning (L1 unstructured, 30% sparsity) (auto-detect num_classes) ──
import os
import torch
import torch.nn.utils.prune as prune

PRUNE_AMOUNT = 0.30  # 30% of weights zeroed per Conv2d layer

def apply_l1_pruning(model, amount=PRUNE_AMOUNT):
    for _, module in model.named_modules():
        if isinstance(module, torch.nn.Conv2d):
            prune.l1_unstructured(module, name='weight', amount=amount)
    return model

def remove_pruning_masks(model):
    for _, module in model.named_modules():
        if isinstance(module, torch.nn.Conv2d):
            if hasattr(module, "weight_orig"):
                prune.remove(module, "weight")
    return model

def compute_conv_weight_sparsity(model):
    nonzero = 0
    total = 0
    for _, module in model.named_modules():
        if isinstance(module, torch.nn.Conv2d):
            w = module.weight.detach()
            nonzero += (w != 0).sum().item()
            total += w.numel()
    return 0.0 if total == 0 else (1.0 - nonzero / total)

def infer_num_classes_from_state_dict(model_name: str, state: dict) -> int:
    if model_name == 'resnet50':
        return state['fc.weight'].shape[0]
    if model_name == 'efficientnet_b0':
        return state['classifier.1.weight'].shape[0]
    if model_name == 'mobilenet_v3_large':
        return state['classifier.3.weight'].shape[0]
    raise ValueError(f"Unknown model_name: {model_name}")

pruning_log = []

for dataset in DATASETS:
    for model_name in CNN_MODELS:
        run_name   = f'{dataset}_{model_name}'
        pth_path   = f'{DRIVE_ROOT}/models/cnn/full/{run_name}_best.pth'
        pruned_out = f'{DRIVE_ROOT}/models/cnn/pruned/{run_name}_pruned.pth'
        os.makedirs(os.path.dirname(pruned_out), exist_ok=True)

        if os.path.exists(pruned_out):
            print(f'[SKIP] Pruned model exists: {pruned_out}')
            continue
        if not os.path.exists(pth_path):
            print(f'[SKIP] Base model not found: {pth_path}')
            continue

        print(f'Pruning {run_name} ({int(PRUNE_AMOUNT*100)}% L1 unstructured on Conv2d)...')

        # Load checkpoint first, infer correct num_classes
        state = torch.load(pth_path, map_location='cpu')
        num_classes_ckpt = infer_num_classes_from_state_dict(model_name, state)

        # Build model with the checkpoint's class count (prevents mismatch)
        model = build_cnn(model_name, num_classes_ckpt)
        model.load_state_dict(state)
        model.eval()

        # Prune + make permanent
        model = apply_l1_pruning(model, PRUNE_AMOUNT)
        model = remove_pruning_masks(model)

        sparsity = compute_conv_weight_sparsity(model)

        torch.save(model.state_dict(), pruned_out)
        size_mb = os.path.getsize(pruned_out) / 1e6
        orig_mb = os.path.getsize(pth_path) / 1e6

        print(f'  Conv weight sparsity: {sparsity*100:.1f}% | Size: {orig_mb:.1f} MB → {size_mb:.1f} MB')

        pruning_log.append({
            'name': run_name,
            'prune_amount': PRUNE_AMOUNT,
            'num_classes_ckpt': int(num_classes_ckpt),
            'conv_weight_sparsity': round(sparsity, 4),
            'orig_mb': round(orig_mb, 2),
            'pruned_mb': round(size_mb, 2),
        })

print('Pruning complete.')


Pruning NEU-DET_resnet50 (30% L1 unstructured on Conv2d)...
  Conv weight sparsity: 30.0% | Size: 94.4 MB → 94.4 MB
Pruning NEU-DET_efficientnet_b0 (30% L1 unstructured on Conv2d)...
  Conv weight sparsity: 30.0% | Size: 16.4 MB → 16.4 MB
Pruning NEU-DET_mobilenet_v3_large (30% L1 unstructured on Conv2d)...
  Conv weight sparsity: 30.0% | Size: 17.0 MB → 17.0 MB
Pruning DAGM_resnet50 (30% L1 unstructured on Conv2d)...
  Conv weight sparsity: 30.0% | Size: 94.4 MB → 94.4 MB
Pruning DAGM_efficientnet_b0 (30% L1 unstructured on Conv2d)...
  Conv weight sparsity: 30.0% | Size: 16.3 MB → 16.3 MB
Pruning DAGM_mobilenet_v3_large (30% L1 unstructured on Conv2d)...
  Conv weight sparsity: 30.0% | Size: 17.0 MB → 17.0 MB
Pruning KSDD2_resnet50 (30% L1 unstructured on Conv2d)...
  Conv weight sparsity: 30.0% | Size: 94.4 MB → 94.4 MB
Pruning KSDD2_efficientnet_b0 (30% L1 unstructured on Conv2d)...
  Conv weight sparsity: 30.0% | Size: 16.3 MB → 16.3 MB
Pruning KSDD2_mobilenet_v3_large (30% L1 uns

In [ ]:
# ── Cell 7: Export Manifest & Summary Table (cleaner + grouped insight) ──
import os
import pandas as pd

def get_file_sizes(base_dir):
    rows = []
    for root, _, files in os.walk(base_dir):
        for fname in files:
            ext = os.path.splitext(fname)[1].lower()
            if ext in ['.onnx', '.tflite', '.pt', '.pth']:
                fpath = os.path.join(root, fname)
                size_mb = os.path.getsize(fpath) / 1e6

                # Try to infer model + variant from filename
                name = os.path.splitext(fname)[0]
                rows.append({
                    'file': name,
                    'path': fpath.replace(base_dir + '/', ''),
                    'format': ext.lstrip('.'),
                    'size_mb': round(size_mb, 2),
                })
    return rows

print('Building export manifest...')

base_models_dir = f'{DRIVE_ROOT}/models'
manifest = get_file_sizes(base_models_dir)
df_manifest = pd.DataFrame(manifest)

# Sort for readability
df_manifest = df_manifest.sort_values(by=['file', 'format']).reset_index(drop=True)

# Save CSV
results_dir = f'{DRIVE_ROOT}/results/tables'
os.makedirs(results_dir, exist_ok=True)
csv_path = f'{results_dir}/optimization_summary.csv'
df_manifest.to_csv(csv_path, index=False)

# Print full table
print(df_manifest.to_string(index=False))

print(f'\nTotal exported files: {len(df_manifest)}')
print(f'Total size across all exports: {df_manifest["size_mb"].sum():.1f} MB')

# Summary by format
print('\nSummary by format:')
format_summary = (
    df_manifest
    .groupby('format')['size_mb']
    .agg(['count', 'sum', 'mean'])
    .round(2)
)
print(format_summary)

# Optional: Summary by model family (resnet / efficientnet / mobilenet)
df_manifest['model_family'] = df_manifest['file'].str.extract(r'(resnet50|efficientnet_b0|mobilenet_v3_large)')
family_summary = (
    df_manifest
    .groupby('model_family')['size_mb']
    .agg(['count', 'sum', 'mean'])
    .round(2)
)

print('\nSummary by model family:')
print(family_summary)

print(f'\nSaved summary CSV to: {csv_path}')


Building export manifest...
                                     file                                                                           path format  size_mb
                     DAGM_efficientnet_b0                                             cnn/onnx/DAGM_efficientnet_b0.onnx   onnx    16.00
                     DAGM_efficientnet_b0                                         cnn/tflite/DAGM_efficientnet_b0.tflite tflite    16.02
                DAGM_efficientnet_b0_best                                         cnn/full/DAGM_efficientnet_b0_best.pth    pth    16.35
      DAGM_efficientnet_b0_fp32_sanitized                              cnn/onnx/DAGM_efficientnet_b0_fp32_sanitized.onnx   onnx    16.52
                DAGM_efficientnet_b0_int8                                   cnn/quantized/DAGM_efficientnet_b0_int8.onnx   onnx     4.54
                DAGM_efficientnet_b0_int8                                 cnn/quantized/DAGM_efficientnet_b0_int8.tflite tflite     4.52
             

In [ ]:
# ── Cell 8: Checkpoint (robust + safe if logs missing) ──
import os
import json
import datetime

# Ensure logs exist even if earlier cells were skipped or partially run
export_log_safe     = export_log if 'export_log' in globals() else []
cnn_export_log_safe = cnn_export_log if 'cnn_export_log' in globals() else []
pruning_log_safe    = pruning_log if 'pruning_log' in globals() else []

checkpoint = {
    'notebook': '04_model_optimization',
    'completed': True,
    'timestamp': datetime.datetime.now().isoformat(),

    # Logs
    'yolo_exports': export_log_safe,
    'cnn_exports': cnn_export_log_safe,
    'pruning': pruning_log_safe,

    # Key artifact
    'manifest_csv': f'{DRIVE_ROOT}/results/tables/optimization_summary.csv',
}

# Save checkpoint
ckpt_dir = f'{DRIVE_ROOT}/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)

ckpt_path = f'{ckpt_dir}/notebook04_status.json'
with open(ckpt_path, 'w') as f:
    json.dump(checkpoint, f, indent=2)

# Clean, readable completion message
print('=' * 60)
print('Notebook 04 Complete')
print(f'Checkpoint saved → {ckpt_path}')
print()

print('Export locations:')
print(f'  YOLO ONNX     → {DRIVE_ROOT}/models/yolo/onnx/')
print(f'  YOLO TFLite   → {DRIVE_ROOT}/models/yolo/tflite/')
print(f'  CNN ONNX      → {DRIVE_ROOT}/models/cnn/onnx/')
print(f'  CNN TFLite    → {DRIVE_ROOT}/models/cnn/tflite/')
print(f'  CNN INT8      → {DRIVE_ROOT}/models/cnn/quantized/')
print(f'  CNN Pruned    → {DRIVE_ROOT}/models/cnn/pruned/')
print('=' * 60)


Notebook 04 Complete
Checkpoint saved → /content/drive/MyDrive/thesis_project/checkpoints/notebook04_status.json

Export locations:
  YOLO ONNX     → /content/drive/MyDrive/thesis_project/models/yolo/onnx/
  YOLO TFLite   → /content/drive/MyDrive/thesis_project/models/yolo/tflite/
  CNN ONNX      → /content/drive/MyDrive/thesis_project/models/cnn/onnx/
  CNN TFLite    → /content/drive/MyDrive/thesis_project/models/cnn/tflite/
  CNN INT8      → /content/drive/MyDrive/thesis_project/models/cnn/quantized/
  CNN Pruned    → /content/drive/MyDrive/thesis_project/models/cnn/pruned/
